# Mineração de candidata — a parte genérica (todas as lições passam por aqui)

**Notebook genérico:** roda sobre qualquer lição (unidade) que passou na triagem, em qualquer base apontada pelo `TRACE`
do `base_pipeline.py`. A lição vem do ambiente: `UNIDADE=<código> uv run jupyter nbconvert --to notebook --execute
--inplace mineracao_generica.ipynb`. Todo texto impresso é calculado do dado; a leitura de cada rodada vive no relatório
dela.

**O que faz:** mede o que vale para qualquer lição — recorrência e escopo (G1), sub-unidades (G2), o que vem antes
(G3) e depois (G4) do erro, estabilidade no tempo (G5) — e grava a população para a parte específica (G6). **Não
extrai o fato nem o hábito:** isso é da parte específica da lição, escolhida pelo funil
(`../docs/16-procedimento-mineracao-candidatas.md`).

**Grava em** `resultados/mineracao/<UNIDADE>/` (git-ignored): `perfil.csv`, `sub_unidades.csv`, `antes.csv`,
`depois.csv`, `estabilidade.csv`, `casos.csv`. O porquê de cada passo: `../docs/06-racionais-mineracao-unidades-n2-n10.md`
§9 (Passos 1, 3, 4) na forma genérica, e o `16`.

## G0 · Funções e carga
Funções numa célula só, sem saída. Entradas: a base compartilhada (`carregar_base()`), o balde invisível
(`resultados/evidencia/silenciosas/ocorrencias.csv` e `casos.csv`, se a base já rodou o notebook das falhas silenciosas).

In [ ]:
import os, re, json
from collections import Counter, defaultdict
import pandas as pd
from base_pipeline import *
import drill_down as dd          # _variante_prompt e _modelo_do_step: as mesmas do `drill_down.py protocolo`

UNIDADE = os.environ.get("UNIDADE")
assert UNIDADE, "defina a lição: UNIDADE=<código da unidade> (ex.: UNIDADE=U_contrato_dict)"
SAIDA = os.path.join("resultados", "mineracao", UNIDADE)
os.makedirs(SAIDA, exist_ok=True)
DEF = re.compile(r"def\s+(\w+)\s*\(")


def ferramentas_chamadas(code, sysprompt):
    """As ferramentas declaradas no system prompt (def nome() que o código do step chama (nome( ) — sem o final_answer."""
    dec = set(DEF.findall(sysprompt or "")) - {"final_answer"}
    return sorted(n for n in dec if re.search(rf"\b{re.escape(n)}\s*\(", code or ""))


def meta_dos_steps(df, execs):
    """(exec_id, role, idx) -> modelo, versão do formato do prompt, chamada do papel e quantas chamadas ele teve."""
    meta = {}
    for _, r in df[df["cod_idef_exeo"].isin(execs)].drop_duplicates("cod_idef_exeo").iterrows():
        try: memo = json.loads(r["txt_etap_memo"])
        except Exception: continue
        for role, lst in memo.items():
            if not isinstance(lst, list): continue
            ch, i = 0, 0
            linhas = []
            for s in lst:
                if isinstance(s, dict) and s.get("__class__") == "TaskStep": ch += 1
                elif isinstance(s, dict) and s.get("__class__") == "ActionStep":
                    linhas.append((i, s, ch)); i += 1
            for i, s, c in linhas:
                meta[(r["cod_idef_exeo"], role, i)] = dict(modelo=dd._modelo_do_step(s),
                                                         versao_prompt=dd._variante_prompt(dd.system_prompt(s)),
                                                         chamada=c, chamadas_do_papel=ch)
    return meta


def triagem_de(linhas, chave):
    """A régua da triagem (MIN_EXECS execuções e MIN_MESES meses) aplicada a cada valor de `chave`."""
    out = []
    for valor, g in linhas.groupby(chave):
        execs, meses = g["exec_id"].nunique(), g["mes"].nunique()
        out.append({"dimensao": chave, "valor": valor, "erros": int((g["canal"] == "visível").sum()),
                    "ocorrencias": g["ocorrencia"].nunique(), "execucoes": execs, "meses": meses,
                    "passa_na_recorrencia": execs >= MIN_EXECS and meses >= MIN_MESES})
    return pd.DataFrame(out).sort_values(["ocorrencias", "erros"], ascending=False)


B = carregar_base()
assert UNIDADE in set(B.EU["unidade"]), f"{UNIDADE} não aparece nos erros desta base"
EU_u = B.EU[B.EU["unidade"] == UNIDADE].copy()
arq_oc = os.path.join("resultados", "evidencia", "silenciosas", "ocorrencias.csv")
S_u = (pd.read_csv(arq_oc, dtype={"exec_id": str}) if os.path.exists(arq_oc) else pd.DataFrame(columns=["unidade"]))
S_u = S_u[S_u["unidade"] == UNIDADE].copy()
RAW = {(r["exec_id"], r["role"], r["idx"]): r for r in B.RAW}
ERRO = {(e.exec_id, e.role, e.idx): e.unidade for e in B.EU.itertuples()}
arq_sil = os.path.join("resultados", "evidencia", "silenciosas", "casos.csv")
SIL = set()
if os.path.exists(arq_sil):
    SIL = {(r.exec_id, r.role, int(r.idx)) for r in pd.read_csv(arq_sil, dtype={"exec_id": str}).itertuples()}
META = meta_dos_steps(B.df, set(EU_u["exec_id"]) | set(S_u.get("exec_id", [])))
TEM_FINAL = dict(zip(B.execs["exec_id"], B.execs["tem_final"]))
FINAL_DO_PAPEL = defaultdict(list)
for r in B.RAW:
    if r["is_final"]: FINAL_DO_PAPEL[(r["exec_id"], r["role"])].append(r["idx"])

# a população da lição: os erros do balde visível + as ocorrências do invisível, no formato comum
V = EU_u[["exec_id", "role", "idx", "mes", "ocorrencia", "submecanismo", "assinatura"]].assign(canal="visível")
S_aux = S_u[["exec_id", "role", "idx", "mes", "ocorrencia"]].assign(canal="silencioso",
          submecanismo="(canal silencioso)", assinatura=S_u.get("grupo", pd.Series(dtype=str)).fillna("(canal silencioso)"))
L = pd.concat([V, S_aux], ignore_index=True)
L["idx"] = L["idx"].astype(int)
L["ferramentas"] = [", ".join(ferramentas_chamadas(RAW.get(k, {}).get("code"), RAW.get(k, {}).get("sysprompt")))
                    or "(nenhuma)" for k in zip(L["exec_id"], L["role"], L["idx"])]
for campo in ("modelo", "versao_prompt", "chamada", "chamadas_do_papel"):
    L[campo] = [META.get(k, {}).get(campo) for k in zip(L["exec_id"], L["role"], L["idx"])]
print(f"lição {UNIDADE} · {UNI[UNIDADE][0]} ({UNI[UNIDADE][1]}) · {len(V)} erros no balde visível · "
      f"{len(S_u)} linhas no invisível")

## G1 · Recorrência e escopo
**O que faz:** conta a lição nos dois canais — erros (só o visível), ocorrências (cascata × unidade, a régua da triagem),
execuções, meses, papéis — e onde ela mora: modelos e ferramentas chamadas no step do erro. **Como ler:** os números de
recorrência são os mesmos da triagem (`candidatos_memoria.csv`); o escopo diz para quem a memória valeria.

In [ ]:
perfil = {"unidade": UNIDADE, "nome": UNI[UNIDADE][0], "tipo": UNI[UNIDADE][1],
          "erros": len(V), "ocorrencias_visiveis": V["ocorrencia"].nunique(), "ocorrencias_silenciosas": S_u["ocorrencia"].nunique(),
          "ocorrencias": V["ocorrencia"].nunique() + S_u["ocorrencia"].nunique(),
          "execucoes": L["exec_id"].nunique(), "meses": L["mes"].nunique(), "papeis": L["role"].nunique(),
          "meses_lista": ";".join(sorted(L["mes"].unique())), "papeis_lista": ";".join(sorted(L["role"].unique()))}
pd.DataFrame([{"medida": k, "valor": v} for k, v in perfil.items()]).to_csv(os.path.join(SAIDA, "perfil.csv"), index=False)
display(pd.DataFrame([perfil]).T.rename(columns={0: "valor"}))
display(L["modelo"].value_counts().rename("erros/ocorrências por modelo").to_frame())
display(L["ferramentas"].value_counts().rename("ferramentas chamadas no step").to_frame())

## G2 · Sub-unidades — a lição é uma só, ou esconde partes diferentes?
**O que faz:** aplica a régua da triagem (MIN_EXECS execuções e MIN_MESES meses) à lição dividida por mecanismo
(submecanismo da taxonomia), por assinatura da mensagem, por papel e por ferramenta chamada no step do erro (o Passo 1
do `06` §9, na forma genérica). Um step que chama várias ferramentas conta
para cada uma (a soma da coluna passa do total). **Como ler:** uma sub-unidade que passa sozinha pode ser uma lição
própria; a parte específica decide. **A ferramenta chamada no step do erro não é a origem do valor:** num erro de
contrato de retorno, o agente usa um valor que veio de uma chamada anterior. Na base 1, a nº2 tem a origem rastreada
na `get_available_documents` (91/96, notebook específico §11.1), e no step do erro a mais chamada é outra. A origem é
da parte específica.

In [ ]:
L_ferr = L.assign(ferramenta_no_step=L["ferramentas"].str.split(", ")).explode("ferramenta_no_step")
SUB = pd.concat([triagem_de(L, "submecanismo"), triagem_de(L, "assinatura"), triagem_de(L, "role"),
                 triagem_de(L_ferr, "ferramenta_no_step")], ignore_index=True)
SUB.to_csv(os.path.join(SAIDA, "sub_unidades.csv"), index=False)
display(SUB)

## G3 · O que vem antes do erro
**O que faz:** para cada erro do balde visível, o step anterior do mesmo papel — teve erro? de qual lição? teve falha
silenciosa de ferramenta? — e se o erro está fora da 1ª chamada do papel. **Como ler:** muito "logo após outro erro"
sugere que a lição é efeito de outra (cascata); falha silenciosa antes sugere que a causa está na ferramenta.

In [ ]:
def antes_de(k):
    e, r, i = k
    if i == 0: return "primeiro step do papel"
    if (e, r, i - 1) in ERRO: return "erro antes: " + ("a mesma lição" if ERRO[(e, r, i - 1)] == UNIDADE else ERRO[(e, r, i - 1)])
    if (e, r, i - 1) in SIL: return "falha silenciosa de ferramenta antes"
    return "step anterior sem erro"
Lv = L[L["canal"] == "visível"].copy()
Lv["antes"] = [antes_de(k) for k in zip(Lv["exec_id"], Lv["role"], Lv["idx"])]
ANTES = Lv["antes"].value_counts().rename_axis("antes").reset_index(name="erros")
fora = int((Lv["chamada"].fillna(1) > 1).sum())
ANTES.to_csv(os.path.join(SAIDA, "antes.csv"), index=False)
display(ANTES)
print(f"erros fora da 1ª chamada do papel: {fora} de {len(Lv)}")

## G4 · O que vem depois do erro
**O que faz:** o step seguinte do mesmo papel (sem erro, a mesma lição de novo, outra lição), se o papel ainda entregou
resposta final e se a execução terminou com resposta. **Como ler:** o agente que se corrige sozinho no step seguinte
precisa menos da memória do que o que repete; a lição que mata a execução pesa mais (o Passo 3 do `06` §9, genérico).

In [ ]:
def depois_de(k):
    e, r, i = k
    if (e, r, i + 1) not in RAW: return "último step do papel"
    if (e, r, i + 1) in ERRO: return "repetiu a mesma lição" if ERRO[(e, r, i + 1)] == UNIDADE else "outro erro: " + ERRO[(e, r, i + 1)]
    return "step seguinte sem erro"
Lv["depois"] = [depois_de(k) for k in zip(Lv["exec_id"], Lv["role"], Lv["idx"])]
Lv["papel_entregou_depois"] = [any(j > i for j in FINAL_DO_PAPEL[(e, r)]) for e, r, i in zip(Lv["exec_id"], Lv["role"], Lv["idx"])]
Lv["execucao_com_resposta"] = [bool(TEM_FINAL.get(e)) for e in Lv["exec_id"]]
DEPOIS = Lv["depois"].value_counts().rename_axis("depois").reset_index(name="erros")
DEPOIS.to_csv(os.path.join(SAIDA, "depois.csv"), index=False)
display(DEPOIS)
print(f"o papel entregou resposta final depois: {int(Lv['papel_entregou_depois'].sum())} de {len(Lv)} · "
      f"a execução terminou com resposta: {int(Lv['execucao_com_resposta'].sum())} de {len(Lv)}")

## G5 · Estabilidade — a lição está no tempo todo, ou concentrada?
**O que faz:** a lição por mês, por versão do formato do prompt e por modelo (o Passo 4 do `06` §9, genérico).
**Como ler:** concentrada num mês, numa versão ou num modelo sugere incidente (não-memória); espalhada sugere hábito ou
fato estável.

In [ ]:
EST = pd.concat([L.groupby(d).agg(ocorrencias=("ocorrencia", "nunique"), execucoes=("exec_id", "nunique"))
                 .reset_index().rename(columns={d: "valor"}).assign(dimensao=d)
                 for d in ("mes", "versao_prompt", "modelo")], ignore_index=True)[["dimensao", "valor", "ocorrencias", "execucoes"]]
EST.to_csv(os.path.join(SAIDA, "estabilidade.csv"), index=False)
display(EST)
for d in ("mes", "versao_prompt", "modelo"):
    t = EST[EST["dimensao"] == d]
    print(f"{d}: {len(t)} valores · o maior concentra {t['ocorrencias'].max() / max(t['ocorrencias'].sum(), 1):.0%} das ocorrências")

## G6 · A população para a parte específica
**O que faz:** grava `casos.csv` — uma linha por erro (visível) ou ocorrência (invisível), com mês, canal, modelo, versão
do prompt, ferramentas, antes, depois e chamada. **Como ler:** é a entrada da parte específica e do
`montar_evidencia.py` do kit (casos por regra fixa, com o cru). Tem `exec_id`: não sai da máquina.

In [ ]:
CASOS = L.merge(Lv[["exec_id", "role", "idx", "antes", "depois", "papel_entregou_depois", "execucao_com_resposta"]],
                on=["exec_id", "role", "idx"], how="left").sort_values(["mes", "exec_id", "idx"])
CASOS.to_csv(os.path.join(SAIDA, "casos.csv"), index=False)
print(f"{len(CASOS)} linhas em {os.path.join(SAIDA, 'casos.csv')} — não sai da máquina")